# Task 1
We will be importing the news sample with pandas, where we can use the read_csv function to get the sample into a data frame

In [20]:
import pandas as pd
import swifter
import re
from rich.progress import track
import nltk
import Stemmer
raw_data = pd.read_csv('../Group_project/news_sample.csv')
print(raw_data['type'])

0      unreliable
1            fake
2      unreliable
3      unreliable
4       clickbait
          ...    
245          fake
246          fake
247           NaN
248           NaN
249           NaN
Name: type, Length: 250, dtype: str


Having imported the data set we can create a function that cleans the data, here we can use the pandas apply() function, so the only function we need is a function that given a series cleans the data to the required specification. Additionally i have removed the comma at the beginning of the sample to remove the unnamed row

In [8]:
EMAIL_REGEX = re.compile(r'\\w+\\w+@\\w+\\.\\w+')
URL_REGEX1 = re.compile(r'https?:((//)|(\\\\))+[^, ]+')
URL_REGEX2 = re.compile(r'www\\.[^, ]+')
NUM_REGEX = re.compile(r'[0-9]+')
DATE_REGEX = re.compile(r'<NUM>-<NUM>-<NUM>.?<NUM>:<NUM>:<NUM>.?<NUM>')

def clean_string(text, **kwargs):
    text = str(text)

    text = text.lower().replace('\n', '').strip()
    
    text = EMAIL_REGEX.sub('<EMAIL>', text)
    text = URL_REGEX1.sub('<URL>', text)
    text = URL_REGEX2.sub('<URL>', text)
    text = NUM_REGEX.sub('<NUM>', text)
    text = DATE_REGEX.sub('<DATE>', text)
    
    return text

def clean_df(df):
    c_df = pd.DataFrame({})
    total = 0
    for i in track(df.columns, 'processing...'):
        c_df[i] = df[i].swifter.progress_bar(False).apply(clean_string)
        total += 1
    return c_df

cleaned_data = clean_df(raw_data)
print(cleaned_data)


Output()

        id                domain        type    url  \
0    <NUM>               awm.com  unreliable  <URL>   
1    <NUM>     beforeitsnews.com        fake  <URL>   
2    <NUM>           cnnnext.com  unreliable  <URL>   
3    <NUM>               awm.com  unreliable  <URL>   
4    <NUM>  bipartisanreport.com   clickbait  <URL>   
..     ...                   ...         ...    ...   
245  <NUM>     beforeitsnews.com        fake  <URL>   
246  <NUM>     beforeitsnews.com        fake  <URL>   
247  <NUM>       www.newsmax.com         nan  <URL>   
248  <NUM>       www.newsmax.com         nan  <URL>   
249  <NUM>       www.newsmax.com         nan  <URL>   

                                               content scraped_at inserted_at  \
0    sometimes the power of christmas will make you...     <DATE>      <DATE>   
1    awakening of <NUM> strands of dna – “reconnect...     <DATE>      <DATE>   
2    never hike alone: a friday the <NUM>th fan fil...     <DATE>      <DATE>   
3    when a rar

We have now cleaned the data and we can move on to tokenizing the text

In [9]:
def tokenize_df(df):
    tk_df = pd.DataFrame({})
    total = 0
    for i in track(df.columns, 'processing...'):
        tk_df[i] = df[i].swifter.progress_bar(False).apply(
            lambda x, **kwargs: nltk.word_tokenize(str(x)))
        total += 1
    return tk_df

tokenized = tokenize_df(cleaned_data)
print(tokenized)

Output()

              id                  domain          type          url  \
0    [<, NUM, >]               [awm.com]  [unreliable]  [<, URL, >]   
1    [<, NUM, >]     [beforeitsnews.com]        [fake]  [<, URL, >]   
2    [<, NUM, >]           [cnnnext.com]  [unreliable]  [<, URL, >]   
3    [<, NUM, >]               [awm.com]  [unreliable]  [<, URL, >]   
4    [<, NUM, >]  [bipartisanreport.com]   [clickbait]  [<, URL, >]   
..           ...                     ...           ...          ...   
245  [<, NUM, >]     [beforeitsnews.com]        [fake]  [<, URL, >]   
246  [<, NUM, >]     [beforeitsnews.com]        [fake]  [<, URL, >]   
247  [<, NUM, >]       [www.newsmax.com]         [nan]  [<, URL, >]   
248  [<, NUM, >]       [www.newsmax.com]         [nan]  [<, URL, >]   
249  [<, NUM, >]       [www.newsmax.com]         [nan]  [<, URL, >]   

                                               content    scraped_at  \
0    [sometimes, the, power, of, christmas, will, m...  [<, DATE, >]   
1  

In [10]:
stop_words = frozenset([word for word in (pd.read_table('../Group_project/englishST.txt'))['words']])

def remove_stop_words(df):
    st_df = pd.DataFrame({})
    total = 0
    for i in track(df.columns, 'processing...'):
        st_df[i] = df[i].swifter.progress_bar(False).apply(
            lambda x, **kwargs: [word for word in x if word not in stop_words]
        )
        total += 1
    return st_df

no_stop_words = remove_stop_words(tokenized)
print(no_stop_words)

Output()

              id                  domain          type          url  \
0    [<, NUM, >]               [awm.com]  [unreliable]  [<, URL, >]   
1    [<, NUM, >]     [beforeitsnews.com]        [fake]  [<, URL, >]   
2    [<, NUM, >]           [cnnnext.com]  [unreliable]  [<, URL, >]   
3    [<, NUM, >]               [awm.com]  [unreliable]  [<, URL, >]   
4    [<, NUM, >]  [bipartisanreport.com]   [clickbait]  [<, URL, >]   
..           ...                     ...           ...          ...   
245  [<, NUM, >]     [beforeitsnews.com]        [fake]  [<, URL, >]   
246  [<, NUM, >]     [beforeitsnews.com]        [fake]  [<, URL, >]   
247  [<, NUM, >]       [www.newsmax.com]         [nan]  [<, URL, >]   
248  [<, NUM, >]       [www.newsmax.com]         [nan]  [<, URL, >]   
249  [<, NUM, >]       [www.newsmax.com]         [nan]  [<, URL, >]   

                                               content    scraped_at  \
0    [power, christmas, make, wild, wonderful, thin...  [<, DATE, >]   
1  

In [11]:
def find_vocabulary(df):
    word_pattern = re.compile(r"[a-z]+'?[a-z]*")
    w = {}
    total = 0
    for col in track(df.columns, description='Processing...'):
        for word_list in df[col]:
            for k in word_list:
                if k in w:
                    w[k] += 1
                elif word_pattern.fullmatch(k):
                    w[k] = 1
        total += 1
                
    return w

vocabulary_st = find_vocabulary(tokenized)
print(len(vocabulary_st))
vocabulary_no_st = find_vocabulary(no_stop_words)
print(len(vocabulary_no_st))
print(len(vocabulary_st) - len(vocabulary_no_st))

Output()

Output()

15816


15333
483


In [12]:
w_stemmer = Stemmer.Stemmer('english')

def stem_words(df):
    st_df = pd.DataFrame({})
    total = 0
    for i in track(df.columns, 'processing...'):
        st_df[i] = df[i].swifter.progress_bar(False).apply(
            lambda x, **kwargs: w_stemmer.stemWords(x)
        )
        total += 1
    return st_df

stemmed = stem_words(no_stop_words)

vocabulary_stem = find_vocabulary(stemmed)
print(len(vocabulary_stem))
print(len(vocabulary_no_st) - len(vocabulary_stem))

Output()

Output()

10283
5050


In [13]:
full_dataset = pd.read_csv('../Group_project/995,000_rows.csv', dtype = 'string')
print(full_dataset)


       Unnamed: 0         id               domain        type  \
0             732  7444726.0   nationalreview.com   political   
1            1348  6213642.0    beforeitsnews.com        fake   
2            7119  3867639.0     dailycurrant.com      satire   
3            1518  9560791.0          nytimes.com    reliable   
4            9345  2059625.0  infiniteunknown.net  conspiracy   
...           ...        ...                  ...         ...   
994995       9376    1304673  21stcenturywire.com  conspiracy   
994996       9625    7273484   nationalreview.com   political   
994997       9561    3489380          thesaker.is     unknown   
994998       4293    6315455        express.co.uk       rumor   
994999       5020    2461199      sputniknews.com        bias   

                                                      url  \
0       http://www.nationalreview.com/node/152734/%E2%...   
1       http://beforeitsnews.com/economy/2012/06/the-c...   
2       http://dailycurrant.com/2016

In [14]:
def process_article(text, **kwargs):
    local_stemmer = Stemmer.Stemmer('english')
    text = str(text).lower().replace('\n', '').strip()
    text = EMAIL_REGEX.sub('<EMAIL>', text)
    text = URL_REGEX1.sub('<URL>', text)
    text = URL_REGEX2.sub('<URL>', text)
    text = NUM_REGEX.sub('<NUM>', text)
    text = DATE_REGEX.sub('<DATE>', text)
    
    tokens = nltk.word_tokenize(text)
    filtered_tokens = [w for w in tokens if w not in stop_words]
    return local_stemmer.stemWords(filtered_tokens)

def pre_processing(df):
    processed_df = pd.DataFrame()
    for col in track(df.columns, description='Processing...'):
        processed_df[col] = df[col].swifter.progress_bar(False).apply(process_article)
    return processed_df

pre_processed_df = pre_processing(full_dataset)
print(pre_processed_df)

Output()

         Unnamed: 0                         id  \
0       [<, NUM, >]  [<, NUM, >, ., <, NUM, >]   
1       [<, NUM, >]  [<, NUM, >, ., <, NUM, >]   
2       [<, NUM, >]  [<, NUM, >, ., <, NUM, >]   
3       [<, NUM, >]  [<, NUM, >, ., <, NUM, >]   
4       [<, NUM, >]  [<, NUM, >, ., <, NUM, >]   
...             ...                        ...   
994995  [<, NUM, >]                [<, NUM, >]   
994996  [<, NUM, >]                [<, NUM, >]   
994997  [<, NUM, >]                [<, NUM, >]   
994998  [<, NUM, >]                [<, NUM, >]   
994999  [<, NUM, >]                [<, NUM, >]   

                                domain          type          url  \
0                 [nationalreview.com]       [polit]  [<, URL, >]   
1                  [beforeitsnews.com]        [fake]  [<, URL, >]   
2                   [dailycurrant.com]       [satir]  [<, URL, >]   
3                        [nytimes.com]     [reliabl]  [<, URL, >]   
4                [infiniteunknown.net]  [conspiraci]  

In [15]:
full_vocabulary = find_vocabulary(pre_processed_df)
print(len(full_vocabulary))

Output()

1386873


In [16]:
vocabulary_df = pd.DataFrame.from_dict(full_vocabulary, orient='index', columns=['Amount'])
vocabulary_df = vocabulary_df.sort_values(by = 'Amount', ascending = False)
vocabulary_df.to_csv('vocabulary_995000.csv')
print(vocabulary_df)

              Amount
na           4612324
state        1015838
time          980130
year          912774
peopl         891321
...              ...
escolhi            1
sigaintevyh        1
preenchi           1
pular              1
strugil            1

[1386873 rows x 1 columns]


In [22]:
training_set = pre_processed_df.sample(frac = 0.8)
rest = pre_processed_df.drop(training_set.index)
validation_set = rest.sample(frac = 0.5)
test_set = rest.drop(validation_set.index)

In [18]:
training_set.to_csv('training_set.csv')
validation_set.to_csv('validation_set.csv')
test_set.to_csv('test_set.csv')